# Lab 1 — Pandas: transformações, agregações e joins

> Extraído de `aulas/Aula_2_aula_python_pandas.ipynb` (seções 5 a 8 e exercícios). As referências a números de seção usam a numeração do notebook original.

## Preparação

Este notebook roda sozinho: as células abaixo recriam o estado necessário, vindo da primeira parte.

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

In [ ]:
# --- Criando a partir de um dicionário --- JSON API, arrquivos de configuração, etc.
dados = {
    "produto":    ["Notebook,laptop", "Mouse", "Teclado", "Monitor", "Headset",
                   "Webcam", "HD Externo", "Pendrive", "Cabo USB", "Hub USB"],
    "categoria":  ["Computadores", "Periféricos", "Periféricos", "Computadores", "Áudio",
                   "Periféricos", "Armazenamento", "Armazenamento", "Cabos", "Periféricos"],
    "preco":      [3500, 85, 150, 1200, 250, 320, 400, 60, 25, 120],
    "estoque":    [10, 45, 30, 8, 20, 15, 12, 100, 200, 18],
    "avaliacao":  [4.8, 4.2, 4.5, 4.7, 3.9, 4.1, 4.6, 4.0, None, 4.3]
}

df = pd.DataFrame(dados)
df

---
## 5. 🔧 Transformações e Limpeza
### 5.1 Criando e modificando colunas

In [ ]:
# Nova coluna — valor total em estoque
df["valor_estoque"] = df["preco"] * df["estoque"]

# Coluna com operação condicional (np.where = IF no SQL)
df["caro"] = np.where(df["preco"] > 500, "Sim", "Não")

# Coluna de texto
df["produto_upper"] = df["produto"].str.upper() # lower()

df[["produto", "preco", "estoque", "valor_estoque", "caro", "produto_upper"]].head(5)

,produto,preco,estoque,valor_estoque,caro,produto_upper
0,"Notebook,laptop",3500,10,35000,Sim,"NOTEBOOK,LAPTOP"
1,Mouse,85,45,3825,Não,MOUSE
2,Teclado,150,30,4500,Não,TECLADO
3,Monitor,1200,8,9600,Sim,MONITOR
4,Headset,250,20,5000,Não,HEADSET


### 5.2 `.apply()` — aplicando funções em colunas/linhas
> 💡 No Spark isso é o equivalente às UDFs (User Defined Functions)

In [ ]:
# Aplicando função à coluna inteira
def classificar_preco(preco):
    if preco < 100:
        return "Econômico"
    elif preco < 500:
        return "Intermediário"
    return "Premium"

df["faixa_preco"] = df["preco"].apply(classificar_preco)

# Com lambda — mesma coisa em uma linha
df["preco_com_frete"] = df["preco"].apply(lambda x: x + 15 if x < 200 else x)

df[["produto", "preco", "faixa_preco", "preco_com_frete"]]

,produto,preco,faixa_preco,preco_com_frete
0,"Notebook,laptop",3500,Premium,3500
1,Mouse,85,Econômico,100
2,Teclado,150,Intermediário,165
3,Monitor,1200,Premium,1200
4,Headset,250,Intermediário,250
5,Webcam,320,Intermediário,320
6,HD Externo,400,Intermediário,400
7,Pendrive,60,Econômico,75
8,Cabo USB,25,Econômico,40
9,Hub USB,120,Intermediário,135


In [ ]:
# apply em múltiplas colunas (axis=1 = linha a linha)
df["resumo"] = df.apply(
    lambda row: f"{row['produto']} ({row['categoria']}) — R$ {row['preco']:.2f}",
    axis=1
)
df["resumo"].head(4)

,resumo
0,"Notebook,laptop (Computadores) — R$ 3500.00"
1,Mouse (Periféricos) — R$ 85.00
2,Teclado (Periféricos) — R$ 150.00
3,Monitor (Computadores) — R$ 1200.00


### 5.3 Tratamento de valores nulos

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   produto          10 non-null     object 
 1   categoria        10 non-null     object 
 2   preco            10 non-null     int64  
 3   estoque          10 non-null     int64  
 4   avaliacao        9 non-null      float64
 5   valor_estoque    10 non-null     int64  
 6   caro             10 non-null     object 
 7   produto_upper    10 non-null     object 
 8   faixa_preco      10 non-null     object 
 9   preco_com_frete  10 non-null     int64  
 10  resumo           10 non-null     object 
dtypes: float64(1), int64(4), object(6)
memory usage: 1012.0+ bytes


In [ ]:
print("Antes — nulos em 'avaliacao':", df["avaliacao"].isnull().sum())

# Opção 1: Preencher com um valor fixo
df_ = df.copy()
df_["avaliacao"] = df_["avaliacao"].fillna(0)

# print("Depois — nulos em 'avaliacao':", df["avaliacao"].isnull().sum())
df_[["produto", "avaliacao"]]

Antes — nulos em 'avaliacao': 1


,produto,avaliacao
0,"Notebook,laptop",4.8
1,Mouse,4.2
2,Teclado,4.5
3,Monitor,4.7
4,Headset,3.9
5,Webcam,4.1
6,HD Externo,4.6
7,Pendrive,4.0
8,Cabo USB,0.0
9,Hub USB,4.3


In [ ]:
df = df.copy()

# Opção 2: Preencher com a média  (mais realista)
media = df["avaliacao"].mean()
df["avaliacao"] = df["avaliacao"].fillna(media)

# Opção 3: Remover linhas com nulos
# df_limpo = df.dropna(subset=["avaliacao"])

# print("Depois — nulos em 'avaliacao':", df["avaliacao"].isnull().sum())
df[["produto", "avaliacao"]]

,produto,avaliacao
0,"Notebook,laptop",4.800000
1,Mouse,4.200000
2,Teclado,4.500000
3,Monitor,4.700000
4,Headset,3.900000
5,Webcam,4.100000
6,HD Externo,4.600000
7,Pendrive,4.000000
8,Cabo USB,4.344444
9,Hub USB,4.300000


### 5.4 Tipos de dados e conversão

In [ ]:
print("Tipos atuais:")
print(df.dtypes)

# Convertendo tipos
df["preco"] = df["preco"].astype(float)
df["categoria"] = df["categoria"].astype("category")  # economiza memória

print("\nTipos após conversão:")
print(df[["preco", "categoria"]].dtypes)

Tipos atuais:
produto             object
categoria           object
preco                int64
estoque              int64
avaliacao          float64
valor_estoque        int64
caro                object
produto_upper       object
faixa_preco         object
preco_com_frete      int64
resumo              object
dtype: object

Tipos após conversão:
preco         float64
categoria    category
dtype: object


### 5.5 Renomear e remover colunas

In [ ]:
# Renomear
df = df.rename(columns={"preco": "preco_brl", "avaliacao": "nota"})

# Remover colunas auxiliares que criamos
df = df.drop(columns=["produto_upper", "resumo", "caro"])

print("Colunas atuais:", df.columns.tolist())

Colunas atuais: ['produto', 'categoria', 'preco_brl', 'estoque', 'nota', 'valor_estoque', 'faixa_preco', 'preco_com_frete']


---
## 6. 📊 Agregações e GroupBy
> Este conceito é **idêntico** no Spark: `df.groupBy(...).agg(...)`

In [ ]:
df.head(10)

,produto,categoria,preco_brl,estoque,nota,valor_estoque,faixa_preco,preco_com_frete
0,"Notebook,laptop",Computadores,3500.0,10,4.800000,35000,Premium,3500
1,Mouse,Periféricos,85.0,45,4.200000,3825,Econômico,100
2,Teclado,Periféricos,150.0,30,4.500000,4500,Intermediário,165
3,Monitor,Computadores,1200.0,8,4.700000,9600,Premium,1200
4,Headset,Áudio,250.0,20,3.900000,5000,Intermediário,250
5,Webcam,Periféricos,320.0,15,4.100000,4800,Intermediário,320
6,HD Externo,Armazenamento,400.0,12,4.600000,4800,Intermediário,400
7,Pendrive,Armazenamento,60.0,100,4.000000,6000,Econômico,75
8,Cabo USB,Cabos,25.0,200,4.344444,5000,Econômico,40
9,Hub USB,Periféricos,120.0,18,4.300000,2160,Intermediário,135


In [ ]:
# Aggregações simples por grupo
df.groupby("categoria")["preco_brl"].mean().round(2)

/tmp/ipykernel_4025/486446988.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("categoria")["preco_brl"].mean().round(2)


,preco_brl
categoria,
Armazenamento,230.00
Cabos,25.00
Computadores,2350.00
Periféricos,168.75
Áudio,250.00


In [ ]:
# Múltiplas métricas com .agg()
resumo = df.groupby("categoria").agg(
    qtd_produtos   = ("produto",    "count"),
    preco_medio    = ("preco_brl",  "mean"),
    preco_max      = ("preco_brl",  "max"),
    estoque_total  = ("estoque",    "sum"),
    nota_media     = ("nota",       "mean")
).round(2).reset_index()

resumo

/tmp/ipykernel_4025/3823900220.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resumo = df.groupby("categoria").agg(


,categoria,qtd_produtos,preco_medio,preco_max,estoque_total,nota_media
0,Armazenamento,2,230.00,400.0,112,4.30
1,Cabos,1,25.00,25.0,200,4.34
2,Computadores,2,2350.00,3500.0,18,4.75
3,Periféricos,4,168.75,320.0,108,4.28
4,Áudio,1,250.00,250.0,20,3.90


In [ ]:
# Ordenando o resumo
resumo.sort_values("qtd_produtos", ascending=False)

,categoria,qtd_produtos,preco_medio,preco_max,estoque_total,nota_media
3,Periféricos,4,168.75,320.0,108,4.28
0,Armazenamento,2,230.00,400.0,112,4.30
2,Computadores,2,2350.00,3500.0,18,4.75
1,Cabos,1,25.00,25.0,200,4.34
4,Áudio,1,250.00,250.0,20,3.90


In [ ]:
# GroupBy + transform → mantém o índice original (útil para criar colunas)
df["preco_medio_categoria"] = df.groupby("categoria")["preco_brl"].transform("mean")
df["acima_da_media"] = df["preco_brl"] > df["preco_medio_categoria"]

df[["produto", "categoria", "preco_brl", "preco_medio_categoria", "acima_da_media"]]

/tmp/ipykernel_4025/853305986.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["preco_medio_categoria"] = df.groupby("categoria")["preco_brl"].transform("mean")


,produto,categoria,preco_brl,preco_medio_categoria,acima_da_media
0,"Notebook,laptop",Computadores,3500.0,2350.00,True
1,Mouse,Periféricos,85.0,168.75,False
2,Teclado,Periféricos,150.0,168.75,False
3,Monitor,Computadores,1200.0,2350.00,False
4,Headset,Áudio,250.0,250.00,False
5,Webcam,Periféricos,320.0,168.75,True
6,HD Externo,Armazenamento,400.0,230.00,True
7,Pendrive,Armazenamento,60.0,230.00,False
8,Cabo USB,Cabos,25.0,25.00,False
9,Hub USB,Periféricos,120.0,168.75,False


In [ ]:
# Pivot Table — visão matricial (linhas × colunas)
# (Requer dados com mais variação; este exemplo é ilustrativo)
df["faixa"] = pd.cut(df["preco_brl"],
                     bins=[0, 100, 500, 5000],
                     labels=["Barato", "Médio", "Caro"])

df

,produto,categoria,preco_brl,estoque,nota,valor_estoque,faixa_preco,preco_com_frete,preco_medio_categoria,acima_da_media,faixa
0,"Notebook,laptop",Computadores,3500.0,10,4.800000,35000,Premium,3500,2350.00,True,Caro
1,Mouse,Periféricos,85.0,45,4.200000,3825,Econômico,100,168.75,False,Barato
2,Teclado,Periféricos,150.0,30,4.500000,4500,Intermediário,165,168.75,False,Médio
3,Monitor,Computadores,1200.0,8,4.700000,9600,Premium,1200,2350.00,False,Caro
4,Headset,Áudio,250.0,20,3.900000,5000,Intermediário,250,250.00,False,Médio
5,Webcam,Periféricos,320.0,15,4.100000,4800,Intermediário,320,168.75,True,Médio
6,HD Externo,Armazenamento,400.0,12,4.600000,4800,Intermediário,400,230.00,True,Médio
7,Pendrive,Armazenamento,60.0,100,4.000000,6000,Econômico,75,230.00,False,Barato
8,Cabo USB,Cabos,25.0,200,4.344444,5000,Econômico,40,25.00,False,Barato
9,Hub USB,Periféricos,120.0,18,4.300000,2160,Intermediário,135,168.75,False,Médio


In [ ]:
pd.pivot_table(df,
               values="estoque", # Colunas para fazer operações
               index=["categoria"], # pivot: colunas para agrupar (referencia)
               columns="faixa", # utilizar como informações
               aggfunc="sum",
               fill_value=0)

/tmp/ipykernel_4025/3791295954.py:1: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pd.pivot_table(df,


faixa,Barato,Médio,Caro
categoria,,,
Armazenamento,100,12,0
Cabos,200,0,0
Computadores,0,0,18
Periféricos,45,63,0
Áudio,0,20,0


---
## 7. 🔗 Joins e Combinações
> No Spark: `df1.join(df2, on="chave", how="left")`

In [ ]:
# Criando um segundo DataFrame de fornecedores
fornecedores = pd.DataFrame({
    "categoria": ["Computadores", "Periféricos", "Áudio", "Armazenamento"],
    "fornecedor": ["TechPro", "OfficeMax", "SoundWave", "DataStore"],
    "prazo_entrega_dias": [7, 3, 5, 4]
})

fornecedores

,categoria,fornecedor,prazo_entrega_dias
0,Computadores,TechPro,7
1,Periféricos,OfficeMax,3
2,Áudio,SoundWave,5
3,Armazenamento,DataStore,4


In [ ]:
df

,produto,categoria,preco_brl,estoque,nota,valor_estoque,faixa_preco,preco_com_frete,preco_medio_categoria,acima_da_media,faixa
0,"Notebook,laptop",Computadores,3500.0,10,4.800000,35000,Premium,3500,2350.00,True,Caro
1,Mouse,Periféricos,85.0,45,4.200000,3825,Econômico,100,168.75,False,Barato
2,Teclado,Periféricos,150.0,30,4.500000,4500,Intermediário,165,168.75,False,Médio
3,Monitor,Computadores,1200.0,8,4.700000,9600,Premium,1200,2350.00,False,Caro
4,Headset,Áudio,250.0,20,3.900000,5000,Intermediário,250,250.00,False,Médio
5,Webcam,Periféricos,320.0,15,4.100000,4800,Intermediário,320,168.75,True,Médio
6,HD Externo,Armazenamento,400.0,12,4.600000,4800,Intermediário,400,230.00,True,Médio
7,Pendrive,Armazenamento,60.0,100,4.000000,6000,Econômico,75,230.00,False,Barato
8,Cabo USB,Cabos,25.0,200,4.344444,5000,Econômico,40,25.00,False,Barato
9,Hub USB,Periféricos,120.0,18,4.300000,2160,Intermediário,135,168.75,False,Médio


In [ ]:
# INNER JOIN — apenas correspondências
df_join = df.merge(fornecedores, on=["categoria"], how="left")
df_join[["produto", "categoria", "preco_brl", "fornecedor", "prazo_entrega_dias"]]

,produto,categoria,preco_brl,fornecedor,prazo_entrega_dias
0,"Notebook,laptop",Computadores,3500.0,TechPro,7.0
1,Mouse,Periféricos,85.0,OfficeMax,3.0
2,Teclado,Periféricos,150.0,OfficeMax,3.0
3,Monitor,Computadores,1200.0,TechPro,7.0
4,Headset,Áudio,250.0,SoundWave,5.0
5,Webcam,Periféricos,320.0,OfficeMax,3.0
6,HD Externo,Armazenamento,400.0,DataStore,4.0
7,Pendrive,Armazenamento,60.0,DataStore,4.0
8,Cabo USB,Cabos,25.0,NaN,NaN
9,Hub USB,Periféricos,120.0,OfficeMax,3.0


In [ ]:
# INNER JOIN — apenas correspondências
df_join = df.merge(fornecedores, on=["categoria"], how="right")
df_join[["produto", "categoria", "preco_brl", "fornecedor", "prazo_entrega_dias"]]

,produto,categoria,preco_brl,fornecedor,prazo_entrega_dias
0,"Notebook,laptop",Computadores,3500.0,TechPro,7
1,Monitor,Computadores,1200.0,TechPro,7
2,Mouse,Periféricos,85.0,OfficeMax,3
3,Teclado,Periféricos,150.0,OfficeMax,3
4,Webcam,Periféricos,320.0,OfficeMax,3
5,Hub USB,Periféricos,120.0,OfficeMax,3
6,Headset,Áudio,250.0,SoundWave,5
7,HD Externo,Armazenamento,400.0,DataStore,4
8,Pendrive,Armazenamento,60.0,DataStore,4


In [ ]:
# INNER JOIN — apenas correspondências
df_join = df.merge(fornecedores, on=["categoria"], how="inner")
df_join[["produto", "categoria", "preco_brl", "fornecedor", "prazo_entrega_dias"]]

,produto,categoria,preco_brl,fornecedor,prazo_entrega_dias
0,"Notebook,laptop",Computadores,3500.0,TechPro,7
1,Mouse,Periféricos,85.0,OfficeMax,3
2,Teclado,Periféricos,150.0,OfficeMax,3
3,Monitor,Computadores,1200.0,TechPro,7
4,Headset,Áudio,250.0,SoundWave,5
5,Webcam,Periféricos,320.0,OfficeMax,3
6,HD Externo,Armazenamento,400.0,DataStore,4
7,Pendrive,Armazenamento,60.0,DataStore,4
8,Hub USB,Periféricos,120.0,OfficeMax,3


In [ ]:
# Tipos de join disponíveis:
# how="inner"  → interseção (padrão)
# how="left"   → tudo da esquerda + match da direita (NaN se não houver)
# how="right"  → tudo da direita + match da esquerda
# how="outer"  → união completa

# Exemplo: LEFT JOIN
df_left = df.merge(fornecedores, on="categoria", how="left")
print("Shape após left join:", df_left.shape)
df_left[["produto", "categoria", "fornecedor"]].head()

Shape após left join: (10, 13)


,produto,categoria,fornecedor
0,"Notebook,laptop",Computadores,TechPro
1,Mouse,Periféricos,OfficeMax
2,Teclado,Periféricos,OfficeMax
3,Monitor,Computadores,TechPro
4,Headset,Áudio,SoundWave


In [ ]:
# pd.concat — empilhando DataFrames (UNION no SQL)
df_eletronicos = df[df["categoria"] == "Computadores"].copy()
df_perifericos = df[df["categoria"] == "Periféricos"].copy()

df_combined = pd.concat([df_eletronicos, df_perifericos], ignore_index=True)
print("Shape combinado:", df_combined.shape)
df_combined[["produto", "categoria"]]

Shape combinado: (6, 11)


,produto,categoria
0,"Notebook,laptop",Computadores
1,Monitor,Computadores
2,Mouse,Periféricos
3,Teclado,Periféricos
4,Webcam,Periféricos
5,Hub USB,Periféricos


In [ ]:
pd.concat([df_eletronicos[["produto", "categoria"]], df_perifericos[["produto", "categoria"]]], ignore_index=True, axis=1)

,0,1,2,3
0,"Notebook,laptop",Computadores,NaN,NaN
3,Monitor,Computadores,NaN,NaN
1,NaN,NaN,Mouse,Periféricos
2,NaN,NaN,Teclado,Periféricos
5,NaN,NaN,Webcam,Periféricos
9,NaN,NaN,Hub USB,Periféricos


---
## 8. Pipeline: Juntando Tudo
> **Encadeamento de operações** (`method chaining`) — código mais limpo e legível.  
> Essa forma de escrever é **idêntica** à usada no PySpark!

In [ ]:
# Recriando o DataFrame base
dados = {
    "produto":    ["Notebook", "Mouse", "Teclado", "Monitor", "Headset",
                   "Webcam", "HD Externo", "Pendrive", "Cabo USB", "Hub USB"],
    "categoria":  ["Computadores", "Periféricos", "Periféricos", "Computadores", "Áudio",
                   "Periféricos", "Armazenamento", "Armazenamento", "Cabos", "Periféricos"],
    "preco":      [3500, 85, 150, 1200, 250, 320, 400, 60, 25, 120],
    "estoque":    [10, 45, 30, 8, 20, 15, 12, 100, 200, 18],
    "avaliacao":  [4.8, 4.2, 4.5, 4.7, 3.9, 4.1, 4.6, 4.0, None, 4.3]
}
df_raw = pd.DataFrame(dados)
df_raw

,produto,categoria,preco,estoque,avaliacao
0,Notebook,Computadores,3500,10,4.8
1,Mouse,Periféricos,85,45,4.2
2,Teclado,Periféricos,150,30,4.5
3,Monitor,Computadores,1200,8,4.7
4,Headset,Áudio,250,20,3.9
5,Webcam,Periféricos,320,15,4.1
6,HD Externo,Armazenamento,400,12,4.6
7,Pendrive,Armazenamento,60,100,4.0
8,Cabo USB,Cabos,25,200,NaN
9,Hub USB,Periféricos,120,18,4.3


In [ ]:
df_ = df_raw.fillna({"avaliacao": df_raw["avaliacao"].mean()}).copy()
df_ = df_.assign(
        valor_estoque = lambda d: d["preco"] * d["estoque"],
        faixa_preco   = lambda d: pd.cut(d["preco"],
                                         bins=[0, 100, 500, 5000],
                                         labels=["Barato", "Médio", "Caro"])
    ).copy()
df_ = df_.query("avaliacao >= 4.0").copy()
df_ = df_[["produto", "categoria", "preco", "estoque", "valor_estoque", "avaliacao", "faixa_preco"]].copy()
df_.sort_values("valor_estoque", ascending=False, inplace=True)
df_.reset_index(drop=True,inplace=True)
df_

,produto,categoria,preco,estoque,valor_estoque,avaliacao,faixa_preco
0,Notebook,Computadores,3500,10,35000,4.800000,Caro
1,Monitor,Computadores,1200,8,9600,4.700000,Caro
2,Pendrive,Armazenamento,60,100,6000,4.000000,Barato
3,Cabo USB,Cabos,25,200,5000,4.344444,Barato
4,Webcam,Periféricos,320,15,4800,4.100000,Médio
5,HD Externo,Armazenamento,400,12,4800,4.600000,Médio
6,Teclado,Periféricos,150,30,4500,4.500000,Médio
7,Mouse,Periféricos,85,45,3825,4.200000,Barato
8,Hub USB,Periféricos,120,18,2160,4.300000,Médio


In [ ]:
# 🚀 Pipeline completo em method chaining
resultado = (
    df_raw
    # 1. Tratar nulos
    .fillna({"avaliacao": df_raw["avaliacao"].mean()})

    # 2. Criar colunas novas
    .assign(
        valor_estoque = lambda d: d["preco"] * d["estoque"],
        faixa_preco   = lambda d: pd.cut(d["preco"],
                                         bins=[0, 100, 500, 5000],
                                         labels=["Barato", "Médio", "Caro"])
    )

    # 3. Filtrar
    .query("avaliacao >= 4.0")

    # 4. Selecionar colunas relevantes
    [["produto", "categoria", "preco", "estoque", "valor_estoque", "avaliacao", "faixa_preco"]]

    # 5. Ordenar
    .sort_values("valor_estoque", ascending=False)

    # 6. Resetar índice
    .reset_index(drop=True)
)

resultado

,produto,categoria,preco,estoque,valor_estoque,avaliacao,faixa_preco
0,Notebook,Computadores,3500,10,35000,4.800000,Caro
1,Monitor,Computadores,1200,8,9600,4.700000,Caro
2,Pendrive,Armazenamento,60,100,6000,4.000000,Barato
3,Cabo USB,Cabos,25,200,5000,4.344444,Barato
4,Webcam,Periféricos,320,15,4800,4.100000,Médio
5,HD Externo,Armazenamento,400,12,4800,4.600000,Médio
6,Teclado,Periféricos,150,30,4500,4.500000,Médio
7,Mouse,Periféricos,85,45,3825,4.200000,Barato
8,Hub USB,Periféricos,120,18,2160,4.300000,Médio


In [ ]:
# Agregação final do pipeline
(
    resultado
    .groupby(["categoria", "faixa_preco"])
    .agg(
        qtd            = ("produto",        "count"),
        valor_total    = ("valor_estoque",  "sum"),
        nota_media     = ("avaliacao",      "mean")
    )
    .round(2)
    .dropna()
    .reset_index()
    .sort_values("valor_total", ascending=False)
)

/tmp/ipykernel_4025/722681050.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["categoria", "faixa_preco"])


,categoria,faixa_preco,qtd,valor_total,nota_media
8,Computadores,Caro,2,44600,4.75
10,Periféricos,Médio,3,11460,4.30
0,Armazenamento,Barato,1,6000,4.00
3,Cabos,Barato,1,5000,4.34
1,Armazenamento,Médio,1,4800,4.60
9,Periféricos,Barato,1,3825,4.20
2,Armazenamento,Caro,0,0,NaN
4,Cabos,Médio,0,0,NaN
7,Computadores,Médio,0,0,NaN
6,Computadores,Barato,0,0,NaN


---
## 🔁 Comparativo: Pandas → Spark

| Conceito | Pandas | PySpark |
|----------|--------|---------|
| Criar DF | `pd.DataFrame(dados)` | `spark.createDataFrame(dados)` |
| Ler CSV | `pd.read_csv("file.csv")` | `spark.read.csv("file.csv", header=True)` |
| Selecionar colunas | `df[["col1", "col2"]]` | `df.select("col1", "col2")` |
| Filtrar | `df[df["preco"] > 100]` | `df.filter(df.preco > 100)` |
| Nova coluna | `df["col"] = ...` | `df.withColumn("col", ...)` |
| GroupBy | `df.groupby("cat").agg(...)` | `df.groupBy("cat").agg(...)` |
| Join | `df.merge(df2, on="k", how="left")` | `df.join(df2, on="k", how="left")` |
| Ordenar | `df.sort_values("col")` | `df.orderBy("col")` |
| Contar linhas | `len(df)` | `df.count()` |
| Mostrar | `df.head(5)` | `df.show(5)` |

> ✅ **Os conceitos são os mesmos — só a sintaxe muda um pouco!**

---
## 🏋️ Exercícios Práticos

Use o `df_raw` criado anteriormente para responder:

#### Exercício 1

Quais produtos têm avaliação acima de 4.4 E estoque maior que 10?
Mostre apenas produto, avaliacao e estoque, ordenado por avaliacao (maior primeiro)

Seu código aqui:


In [ ]:
# Gabarito 1
(
    df_raw
    .dropna(subset=["avaliacao"])
    .query("avaliacao > 4.4 and estoque > 10")
    [["produto", "avaliacao", "estoque"]]
    .sort_values("avaliacao", ascending=False)
)

,produto,avaliacao,estoque
6,HD Externo,4.6,12
2,Teclado,4.5,30


### Exercício 2

Calcule o valor total em estoque (preco * estoque) por categoria
Mostre o resultado ordenado do maior para o menor

Seu código aqui:


In [ ]:
# Gabarito 2
(
    df_raw
    .assign(valor_estoque=lambda d: d["preco"] * d["estoque"])
    .groupby("categoria")["valor_estoque"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"valor_estoque": "valor_total"})
)

,categoria,valor_total
0,Computadores,44600
1,Periféricos,15285
2,Armazenamento,10800
3,Cabos,5000
4,Áudio,5000



### Exercício 3

Crie uma coluna "desconto_10pct" com o preço após 10% de desconto usando .apply() com uma função lambda

Seu código aqui:


In [ ]:
# Gabarito 3
df_ex3 = df_raw.copy()
df_ex3["desconto_10pct"] = df_ex3["preco"].apply(lambda p: round(p * 0.90, 2))
df_ex3[["produto", "preco", "desconto_10pct"]]

,produto,preco,desconto_10pct
0,Notebook,3500,3150.0
1,Mouse,85,76.5
2,Teclado,150,135.0
3,Monitor,1200,1080.0
4,Headset,250,225.0
5,Webcam,320,288.0
6,HD Externo,400,360.0
7,Pendrive,60,54.0
8,Cabo USB,25,22.5
9,Hub USB,120,108.0


### Exercício 4

Em um único pipeline encadeado (method chaining):
1. Preencha nulos de avaliacao com a mediana
2. Crie a coluna valor_estoque = preco * estoque
3. Filtre apenas produtos com valor_estoque > 1000
4. Agrupe por categoria e calcule: soma do valor_estoque e média de avaliacao
5. Ordene por soma do valor_estoque (desc)

Seu código aqui:


In [ ]:
# Gabarito 4
(
    df_raw
    .fillna({"avaliacao": df_raw["avaliacao"].median()})
    .assign(valor_estoque=lambda d: d["preco"] * d["estoque"])
    .query("valor_estoque > 1000")
    .groupby("categoria")
    .agg(
        valor_total  = ("valor_estoque", "sum"),
        nota_media   = ("avaliacao",     "mean")
    )
    .round(2)
    .reset_index()
    .sort_values("valor_total", ascending=False)
)

,categoria,valor_total,nota_media
2,Computadores,44600,4.75
3,Periféricos,15285,4.28
0,Armazenamento,10800,4.30
1,Cabos,5000,4.30
4,Áudio,5000,3.90


---
## 📚 Referências e Próximos Passos

### Documentação
- [Pandas Documentation](https://pandas.pydata.org/docs/)
- [PySpark Documentation](https://spark.apache.org/docs/latest/api/python/)
- [Pandas Cheat Sheet (DataCamp)](https://www.datacamp.com/cheat-sheet/pandas-cheat-sheet-for-data-science-in-python)

### Próximas aulas — Apache Spark 🚀
```
✅ Pandas → DataFrames, transformações, filtros, GroupBy, Joins
⏭️  Spark  → Mesmos conceitos, execução distribuída, lazy evaluation
            → RDDs, DataFrames, Datasets
            → SparkSQL, Delta Lake, streaming
```

### Equivalência conceitual
| O que aprendemos hoje | O que vem no Spark |
|-----------------------|--------------------|
| `pd.DataFrame` | `spark.DataFrame` |
| Execução imediata | **Lazy evaluation** (transforma → ação) |
| 1 máquina | Cluster de máquinas |
| `.apply()` / lambda | UDFs / Pandas UDFs |
| `.read_csv()` | `.read.parquet()` / `.read.delta()` |